# The command line

<svg xmlns="http://www.w3.org/2000/svg" width="480" height="200" viewBox="0 0 600 250">
<rect x="60" y="20" width="480" height="170" rx="14" fill="#2d2f33"/>
<rect x="60" y="20" width="480" height="34" rx="14" fill="#44474d"/>
<rect x="60" y="40" width="480" height="14" fill="#44474d"/>
<circle cx="86" cy="37" r="7" fill="#ff5f57"/><circle cx="110" cy="37" r="7" fill="#febc2e"/><circle cx="134" cy="37" r="7" fill="#28c840"/>
<text x="92" y="105" fill="#28c840" font-family="monospace" font-size="34" font-weight="bold">$</text>
<text x="128" y="105" fill="#f2f2f2" font-family="monospace" font-size="34">ls -lh</text>
<text x="92" y="152" fill="#9aa0a6" font-family="monospace" font-size="34">patients.csv</text>
<rect x="92" y="170" width="20" height="5" fill="#28c840"/>
<text x="300" y="235" fill="#2d2f33" font-family="Helvetica, Arial, sans-serif" font-size="26" font-weight="bold" text-anchor="middle">The shell</text>
</svg>

[Each question is followed by a cell that already starts with `%%bash`: write your command there, under the lines that are given. Everything in such a cell is run by bash, not by Python.]

To understand what a command does, paste it into [explainshell.com](https://explainshell.com/): it explains each part of the command.

Two things to know before starting:

- Every cell is a **new shell**, so a `cd` in one cell is forgotten in the next. That is why the cells working on the data start again with `cd ../data`.
- The data is the folder `../data`, one level up: a synthetic export from a hospital ward (`patients.csv`, one ECG trace per file in `ecg/`, free text notes in `notes/`, and the log of the export in `export.log`).

Run the cell below first. It is the only **Python** cell of the notebook: it builds `../data` if the folder is not there yet. Everything is synthetic and seeded, so it is the same export for everybody.

In [1]:
from pathlib import Path
from subprocess import run
from sys import executable

data = Path("../data")
if (data / "patients.csv").exists():
    print(f"data already there: {data.resolve()}")
else:
    run([executable, "../../scripts/make_data_c01.py", str(data)], check=True)
    print(f"data generated in {data.resolve()}")

data already there: /home/paul/Documents/Cours/MastereSpecialiseSanteNumerique2026/Programming01/data


## 1. Where am I, and what is here?

Print the folder you are in (`pwd`), then list what sits next to this notebook (`ls`).

In [2]:
%%bash
pwd
ls

/home/paul/Documents/Cours/MastereSpecialiseSanteNumerique2026/Programming01/Session03
clean.py
command_line.ipynb
command_line-solutions.ipynb
img
rows.py
slides.md
slides.pptx


Now list `../data`, one file per line, with sizes a human can read, and with the hidden files (`-l`, `-h`, `-a`).

In [3]:
%%bash
cd ../data
ls -lha

total 44K
drwxrwxr-x  5 paul paul 4.0K Oct 10 19:30 .
drwxrwxr-x 10 paul paul 4.0K Oct 13 14:33 ..
drwxrwxr-x  2 paul paul 4.0K Sep  8 12:30 ecg
-rw-rw-r--  1 paul paul 2.0K Sep  8 12:30 export.log
drwxrwxr-x  2 paul paul 4.0K Sep  8 12:30 exports
-rw-rw-r--  1 paul paul    1 Oct 10 19:30 .gitignore
drwxrwxr-x  2 paul paul 4.0K Sep  8 12:30 notes
-rw-rw-r--  1 paul paul  11K Sep  8 12:30 patients.csv
-rw-rw-r--  1 paul paul  129 Sep  8 12:30 README.md


How many files are there in `ecg/`?

*Hint: `ls` prints one name per line when its output is not a terminal, and `wc -l` counts lines.*

In [4]:
%%bash
cd ../data
ls ecg/ | wc -l

60


## 2. Looking inside a file

Count the lines of `patients.csv` (`wc -l`), then print its first three rows and its last row (`head`, `tail`).

There are 240 patients. Why does `wc -l` disagree?

In [5]:
%%bash
cd ../data
wc -l patients.csv
head -3 patients.csv
tail -1 patients.csv  # 241 lines: the header is a line too

241 patients.csv
patient_id,age,sex,ward,admission,los_days,outcome
P0001,66,F,ICU,2024-04-21,6.4,transferred
P0002,14,F,ICU,2024-04-14,10.6,transferred
P0240,,F,CARDIO,2024-12-10,1.5,discharged


`cat` prints a whole file. Print the note `notes/P0001.txt` with `cat`.

Then run `cat ecg/P0002.csv | wc -l`: how many lines would `cat` have printed into the notebook?

In [6]:
%%bash
cd ../data
cat notes/P0001.txt
cat ecg/P0002.csv | wc -l  # cat is fine for 10 lines, not for a signal

Ward round note for P0001.
Escalated to ICU overnight, sepsis suspected.
121


## 3. Keeping the lines you want: `grep`

How many rows of `patients.csv` mention `ICU` (`-c`)? Print three of them as a check.

In [7]:
%%bash
cd ../data
grep -c 'ICU' patients.csv
grep 'ICU' patients.csv | head -3

76
P0001,66,F,ICU,2024-04-21,6.4,transferred
P0002,14,F,ICU,2024-04-14,10.6,transferred
P0003,73,M,ICU,2024-07-09,8.6,transferred


How many **CARDIO** stays ended with the patient transferred? Chain two `grep` with a pipe: the first narrows, the second counts what is left.

In [8]:
%%bash
cd ../data
grep 'CARDIO' patients.csv | grep -c 'transferred'

16


Which ward round notes in `notes/` mention sepsis, whatever the case? Print the file names only.

*Hint: `-r` searches a whole folder, `-i` ignores case, `-l` prints names instead of matching lines.*

In [9]:
%%bash
cd ../data
grep -rli 'sepsis' notes/

notes/P0012.txt
notes/P0001.txt
notes/P0006.txt
notes/P0005.txt


## 4. Columns, order, counts: `cut`, `sort`, `uniq`

Print the ward of the first five patients.

*Hint: `cut` needs the separator (`-d,`) and the field number (`-f4`); fields are numbered from 1.*

In [10]:
%%bash
cd ../data
cut -d, -f4 patients.csv | head -5

ward
ICU
ICU
ICU
PEDIA


Count the stays per ward, most frequent first.

*Hint: `sort` puts identical lines next to each other, `uniq -c` collapses and counts them, and `sort -rn` orders the counts. Drop the header first with `tail -n +2`, which means "start at line 2".*

In [11]:
%%bash
cd ../data
tail -n +2 patients.csv | cut -d, -f4 | sort | uniq -c | sort -rn

     76 ICU
     64 CARDIO
     39 ONCO
     35 PEDIA
     26 GERIA


Print the five longest stays, longest first.

*Hint: `sort` can work on a column of its own: `-t,` sets the separator, `-k6` sorts on field 6, `-rn` orders it as a number, in reverse.*

In [12]:
%%bash
cd ../data
tail -n +2 patients.csv | sort -t, -k6 -rn | head -5

P0221,59,F,ICU,2024-10-19,18.0,discharged
P0030,73,M,ICU,2024-02-14,17.4,discharged
P0174,62,F,ICU,2024-08-20,16.0,deceased
P0077,83,M,ICU,2024-11-06,16.0,deceased
P0192,63,M,ICU,2024-02-14,15.4,deceased


## 5. Finding files

How many `.csv` files are there under `../data`, at any depth? Print five of them.

*Hint: `find . -name '*.csv'` walks the tree from here down. The quotes matter: without them, the shell expands `*.csv` before `find` ever sees it.*

In [13]:
%%bash
cd ../data
find . -name '*.csv' | wc -l
find . -name '*.csv' | head -5

62
./ecg/P0048.csv
./ecg/P0030.csv
./ecg/P0060.csv
./ecg/P0003.csv
./ecg/P0007.csv


One ECG trace arrived empty from the recorder. Find it (`-type f`, `-empty`).

In [14]:
%%bash
cd ../data
find . -type f -empty

./ecg/P0018.csv


The folder `exports/` holds a file whose name contains spaces. Print its first two lines.

*Hint: a space separates arguments, so the name has to be quoted, or the shell reads it as three arguments.*

In [15]:
%%bash
cd ../data
head -2 'exports/Patient Records 2024.csv'

patient_id,age,sex,ward,admission,los_days,outcome
P0001,66,F,ICU,2024-04-21,6.4,transferred


## 6. Writing the results down

Write the table of stays per ward into `../results/wards.txt`, then print the file to check it.

*Hint: `>` sends the output to a file instead of the screen, and `mkdir -p` creates the folder (and does not complain if it is already there).*

In [16]:
%%bash
cd ../data
mkdir -p ../results
tail -n +2 patients.csv | cut -d, -f4 | sort | uniq -c | sort -rn > ../results/wards.txt
cat ../results/wards.txt

     76 ICU
     64 CARDIO
     39 ONCO
     35 PEDIA
     26 GERIA


Write the ICU rows of `patients.csv` into `../results/icu.csv`, with the header on the first line. Then check that the file has the number of lines you expect.

*Hint: `>` overwrites, `>>` appends.*

In [17]:
%%bash
cd ../data
head -1 patients.csv > ../results/icu.csv
grep 'ICU' patients.csv >> ../results/icu.csv
wc -l ../results/icu.csv

77 ../results/icu.csv


## 7. Calling Python from the shell

Where is the `python3` program, and which version is it? Then let bash start it on a one line program.

*Hint: `which`, `python3 --version`, and `python3 -c '...'`. Single quotes hand the program to Python untouched.*

In [18]:
%%bash
which python3
python3 --version
python3 -c 'print("hello from python, started by bash")'

/home/paul/Documents/Cours/MastereSpecialiseSanteNumerique2026/.venv/bin/python3
Python 3.14.8
hello from python, started by bash
